# Демо проектных упражнений модуля 13 — От диалогов к обучению; полный шаг и LoRA

**Демо-реализация, не готовое решение полномасштабного проекта.** Демо 1 выполняет сериализацию, упаковку, проверку градиентов, обучение, генерацию и восстановление малой модели на CPU. Демо 2 изучает ранг адаптера при общей исходной матрице. Все данные синтетические, сеть и ключи не нужны.
Все входы малы, детерминированы и работают без сети на CPU. Цель — проверить
интерфейсы, метрики, инварианты и структуру отчёта до расходования GPU/API-бюджета.

In [ ]:
from __future__ import annotations
import importlib.util, subprocess, sys
for package in ("numpy", "matplotlib"):
    if importlib.util.find_spec(package) is None:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", package])


import math
import random
import time
from collections import Counter, defaultdict

import matplotlib.pyplot as plt
import numpy as np

SEED = 20260801
random.seed(SEED)
np.random.seed(SEED)
print(f"воспроизводимый тестовый запуск; seed={SEED}")
DEMO_LANGUAGE = 'ru'


## Демо 1. От диалога к обучению

Первый опыт связывает подготовку диалога с обучением. После заголовка ассистента находятся три цели: два символа в обратном порядке и END. Перед оптимизацией сравним отдельные и упакованные примеры по логитам, сумме потерь и градиентам. На 56 парах разных символов обучаем модель с нуля; восемь одинаковых пар отложены. Это проверка переноса комбинаций и механики целевых меток. Она не воспроизводит дообучение предобученной LLM.

In [ ]:
"""CPU teaching experiment: serialize, pack, train, generate, and reload.

Synthetic symbol reversal exposes the data contract; it is not evidence about LLM quality.
"""
import copy
import importlib.util
import io
import subprocess
import sys

if importlib.util.find_spec("torch") is None:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "torch"])
import torch
from torch import nn
from torch.nn import functional as F

torch.set_num_threads(1)
torch.manual_seed(130)
VOCAB, WIDTH, BOS, USER, ASSISTANT, END = 16, 32, 0, 1, 2, 3


def dialogue(a, b):
    # Reverse two symbols. Role headers are inputs; answer and END are targets.
    ids = torch.tensor([BOS, USER, a, b, END, ASSISTANT, b, a, END])
    labels = ids.clone()
    labels[:6] = -100
    return ids, labels


def pack(records):
    ids, labels, positions, segments = [], [], [], []
    for i, (x, y) in enumerate(records):
        target = y.clone()
        target[0] = -100  # No prediction across an independent example boundary.
        ids.append(x); labels.append(target)
        positions.append(torch.arange(len(x)))
        segments.append(torch.full_like(x, i))
    return tuple(torch.cat(xs) for xs in (ids, labels, positions, segments))


class TinyCausalLM(nn.Module):
    def __init__(self):
        super().__init__()
        self.token = nn.Embedding(VOCAB, WIDTH)
        self.position = nn.Embedding(16, WIDTH)
        self.qkv = nn.Linear(WIDTH, 3 * WIDTH)
        self.proj = nn.Linear(WIDTH, WIDTH)
        self.norm = nn.LayerNorm(WIDTH)
        self.ff = nn.Sequential(nn.Linear(WIDTH, 64), nn.GELU(), nn.Linear(64, WIDTH))
        self.head = nn.Linear(WIDTH, VOCAB)

    def forward(self, ids, positions, segments):
        h = self.token(ids) + self.position(positions)
        q, k, v = self.qkv(h).chunk(3, dim=-1)
        idx = torch.arange(len(ids))
        allowed = (segments[:, None] == segments[None, :]) & (idx[:, None] >= idx[None, :])
        scores = q @ k.T / WIDTH**.5
        h = self.norm(h + self.proj(scores.masked_fill(~allowed, -torch.inf).softmax(-1) @ v))
        return self.head(h + self.ff(h))


def loss_sum(model, batch):
    x, y, pos, seg = batch
    logits = model(x, pos, seg)
    return F.cross_entropy(logits[:-1], y[1:], ignore_index=-100, reduction="sum")


train = [dialogue(a, b) for a in range(4, 12) for b in range(4, 12) if a != b]
heldout = [dialogue(a, a) for a in range(4, 12)]
model = TinyCausalLM()

# Packing preserves logits, summed loss and gradients when positions and boundaries match.
records = train[:3]
packed_batch = pack(records)
with torch.no_grad():
    packed_logits = model(packed_batch[0], packed_batch[2], packed_batch[3])
    individual = torch.cat([model(b[0], b[2], b[3]) for b in map(lambda r: pack([r]), records)])
torch.testing.assert_close(packed_logits, individual, rtol=1e-5, atol=1e-6)
packed_loss = loss_sum(model, packed_batch)
separate_loss = sum(loss_sum(model, pack([r])) for r in records)
torch.testing.assert_close(packed_loss, separate_loss, rtol=1e-6, atol=1e-6)
gp = torch.autograd.grad(packed_loss, tuple(model.parameters()))
gs = torch.autograd.grad(separate_loss, tuple(model.parameters()))
for left, right in zip(gp, gs):
    torch.testing.assert_close(left, right, rtol=1e-4, atol=3e-6)

def score(model, records):
    with torch.no_grad():
        return sum(float(loss_sum(model, pack([r]))) for r in records) / (3 * len(records))

before = {"train": score(model, train), "heldout": score(model, heldout)}
optimizer = torch.optim.AdamW(model.parameters(), lr=.006)
history = []
for step in range(240):
    # Eight independent examples per packed row; all 56 rotate through training.
    subset = [train[(step * 8 + j) % len(train)] for j in range(8)]
    batch = pack(subset)
    loss = loss_sum(model, batch) / int((batch[1][1:] != -100).sum())
    optimizer.zero_grad(); loss.backward(); optimizer.step()
    history.append(float(loss.detach()))
after = {"train": score(model, train), "heldout": score(model, heldout)}
assert after["train"] < before["train"] / 5

@torch.no_grad()
def generate(model, prompt, max_new=4):
    sequence = prompt.clone()
    output = []
    for _ in range(max_new):
        logits = model(sequence, torch.arange(len(sequence)), torch.zeros_like(sequence))
        token = int(logits[-1].argmax())
        output.append(token)
        sequence = torch.cat([sequence, torch.tensor([token])])
        if token == END:
            break
    return output

# Generate once per held-out example, then derive every metric from stored outputs.
generated = [generate(model, ids[:6]) for ids, _ in heldout]
exact = sum(pred == ids[6:].tolist() for pred, (ids, _) in zip(generated, heldout)) / len(heldout)
stop_rate = sum(bool(pred) and pred[-1] == END for pred in generated) / len(generated)
buffer = io.BytesIO()
torch.save(model.state_dict(), buffer); buffer.seek(0)
restored = TinyCausalLM()
restored.load_state_dict(torch.load(buffer, weights_only=True))
with torch.no_grad():
    probe = pack(heldout[:2])
    torch.testing.assert_close(model(probe[0], probe[2], probe[3]), restored(probe[0], probe[2], probe[3]), rtol=0, atol=0)
print("token NLL before / after:", before, after)
print("held-out exact / stop rate:", exact, stop_rate, "stored outputs:", generated)
print("packed logits, summed loss, gradients and checkpoint reload: PASS")

Совпадение трёх величин до обучения проверяет изоляцию сильнее, чем один взгляд на маску. После обучения читаем отдельно потерю, точность всего ответа и долю корректных остановок: завершившийся ответ может быть ошибочным. Все показатели генерации рассчитаны по одному сохранённому набору ответов. Восстановление весов проверяется на тех же входах; для вывода о переносе на естественный язык этих данных недостаточно.

## Демо 2. Ранг адаптера и ошибка на новых данных

Во втором опыте известно требуемое изменение матрицы: оно имеет ранг четыре. Полное обновление и адаптеры начинают с одной базовой матрицы и получают одни данные. Для удобства вычислений в коде используются векторы-строки; поэтому порядок и формы множителей транспонированы относительно формулы лекции. Масштаб alpha/r равен единице. Один множитель адаптера случайный, другой нулевой, так что начальный прогноз совпадает с базовым, но обучение может начаться.

In [ ]:
# Same base and data for full updates and LoRA; the target change has rank four.
torch.manual_seed(131)
x_train, x_test = torch.randn(256, 32), torch.randn(128, 32)
base = torch.randn(32, 32) / 32**.5
delta = torch.randn(32, 4) @ torch.randn(4, 32) / 8
y_train, y_test = x_train @ (base + delta), x_test @ (base + delta)
results = []
for rank in (None, 2, 4, 8):
    torch.manual_seed(132)
    frozen = base.clone()
    if rank is None:
        update = torch.zeros_like(base, requires_grad=True)
        params = [update]
        predict = lambda x: x @ (frozen + update)
    else:
        a = (torch.randn(32, rank) * .03).requires_grad_()
        b = torch.zeros(rank, 32, requires_grad=True)
        params = [a, b]
        predict = lambda x: x @ frozen + (x @ a) @ b  # alpha/r = 1
    optimizer = torch.optim.Adam(params, lr=.05)
    for _ in range(240):
        loss = (predict(x_train) - y_train).square().mean()
        optimizer.zero_grad(); loss.backward(); optimizer.step()
    with torch.no_grad():
        train_mse = float((predict(x_train)-y_train).square().mean())
        test_predictions = predict(x_test)
        test_mse = float((test_predictions-y_test).square().mean())
    assert torch.equal(frozen, base)
    assert test_predictions.shape == y_test.shape
    assert torch.isfinite(test_predictions).all()
    assert torch.isfinite(torch.tensor([train_mse, test_mse])).all()
    results.append(dict(rank=rank, trainable=sum(p.numel() for p in params),
                        train_mse=train_mse, test_mse=test_mse))
for result in results: print(result)
comparison = results[2]['test_mse'] < results[1]['test_mse']
label = ('Ошибка ранга 4 меньше ошибки ранга 2 в этом запуске:' if DEMO_LANGUAGE == 'ru'
         else 'Rank 4 has lower held-out error than rank 2 in this run:')
print(label, comparison)
print('This is a matrix adaptation experiment, not a measured GPU-memory or LLM-quality comparison.')

Полное обновление содержит 1024 параметра, адаптеры рангов 2, 4 и 8 — 128, 256 и 512. Ранг два не может точно выразить заданное изменение ранга четыре; ранги четыре и восемь имеют достаточную ёмкость, хотя конечная оптимизация не обязана достигать точного решения. Ошибка на отложенных входах показывает результат именно этого запуска. Меняйте число обучающих примеров и истинный ранг обновления, затем сопоставляйте ошибки: при малом объёме данных более ёмкий адаптер может оказаться хуже на новых входах. Такой порядок результатов допустим; проверяются формы, конечность вычислений и неизменность базовой матрицы. Время и память здесь не измеряются, а сильные результаты адаптера на специально построенной матрице не означают его универсального преимущества.

## Переход к полному проекту

В полном проекте закрепите официальный токенизатор и шаблон, разделите семейства задач до генерации синтетики, проверьте усреднение по токенам при накоплении градиентов. Сравнивайте полное дообучение и LoRA на одной базе, с удержанными задачами и измеренными временем и памятью.

### Минимальный отчёт

1. Версии данных, модели, кода и seed.
2. Основная метрика, а также качество, стоимость, задержка и память там, где это применимо.
3. Базовая линия, одинаковый бюджет и хотя бы один отрицательный контроль.
4. Неудачные прогоны и границы обоснованных выводов.